In [10]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import keras 
from keras.models import Sequential 
from keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

In [11]:

df = pd.read_pickle(r"C:\Users\valem\OneDrive\Documenti\weather_forecast\dfs.pickle")
print(df)

df['temp_mean_past1h'] = df['temp_mean_past1h'].fillna(0)
temperature = df["temp_mean_past1h"].values.reshape(-1, 1)
df['wind_speed_past1h'] = df['wind_speed_past1h'].fillna(0)
wind_speed = df["wind_speed_past1h"].values.reshape(-1, 1)

                      temp_mean_past1h  wind_speed_past1h
2025-10-09T22:00:00Z              11.8                4.5
2025-10-09T23:00:00Z              11.4                4.3
2025-10-10T00:00:00Z              11.3                4.5
2025-10-10T01:00:00Z              11.2                4.3
2025-10-10T02:00:00Z              11.4                4.3
...                                ...                ...
2025-11-09T18:00:00Z               6.7                2.3
2025-11-09T19:00:00Z               6.5                1.5
2025-11-09T20:00:00Z               5.9                1.3
2025-11-09T21:00:00Z               4.3                1.0
2025-11-09T22:00:00Z               4.3                0.2

[746 rows x 2 columns]


In [12]:
746*0.8

596.8000000000001

In [13]:
#train/test split
train_df, test_df = df[1:596], df[596:]

In [15]:
train = train_df
scalers = {}
for i in train_df.columns:
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaling = scaler.fit_transform(train[i].values.reshape(-1, 1))
    scaling = np.reshape(scaling, len(scaling))
    scalers['scaler_'+i] = scaler
    train[i] = scaling

test = test_df
for i in train_df.columns:
    scaler = scalers['scaler_'+i]
    scaling = scaler.transform(test[i].values.reshape(-1, 1))
    scaling = np.reshape(scaling, len(scaling))
    scalers['scaler_'+i] = scaler
    test[i] = scaling




In [16]:
#creating samples of input past observation / output future observ

def split_series(series, n_past, n_future):
    X, y = list(), list()
    for start in range(len(series)):
        past_end = start + n_past
        future_end = past_end + n_future
        if future_end > len(series):
            break
        past, future = series[start:past_end, :], series[past_end:future_end, :]
        X.append(past)
        y.append(future)
    return np.array(X), np.array(y)